In [ ]:
#手書き計算式を自動生成
import cv2
import numpy as np
import random
import shutil
from pathlib import Path
from tqdm import tqdm

# --- 設定 ---
base_path = Path(r'math_dataset_cnn')
output_path = Path(r'yolo_synthetic_data')
num_images = 3000 #画像枚数

classes = ["equation", "fraction", "bar"]
class_to_id = {name: i for i, name in enumerate(classes)}

def reset_folders():
    if output_path.exists():
        shutil.rmtree(output_path)
    output_path.mkdir(parents=True, exist_ok=True)
    (output_path / "images").mkdir()
    (output_path / "labels").mkdir()

def get_random_char(char_name):
    char_dir = base_path / char_name
    img_pts = list(char_dir.glob("*.png"))
    if not img_pts: return np.full((48,48), 255, dtype=np.uint8)
    img = cv2.imread(str(random.choice(img_pts)), cv2.IMREAD_GRAYSCALE)
    
    angle = random.uniform(-10, 10)
    h, w = img.shape
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    img = cv2.warpAffine(img, M, (w, h), borderValue=255)
    return img

def paste_char(canvas, char_name, x, y, size):
    img = get_random_char(char_name)
    img = cv2.resize(img, (size, size))
    h, w = img.shape
    if y < 0 or y+h > canvas.shape[0] or x < 0 or x+w > canvas.shape[1]:
        return None
    roi = canvas[y:y+h, x:x+w]
    canvas[y:y+h, x:x+w] = cv2.bitwise_and(roi, img)
    return (x, y, w, h)

def generate_mixed_data():
    reset_folders()
    print(f"データ生成: {num_images} 枚")
    
    for idx in tqdm(range(num_images), desc="Progress"):
        c_w, c_h = random.randint(1100, 1500), random.randint(900, 1300)
        # 背景にノイズを追加
        canvas = np.full((c_h, c_w), 255, dtype=np.uint8)
        if random.random() > 0.7:
            noise = np.random.randint(245, 256, (c_h, c_w), dtype=np.uint8)
            canvas = cv2.bitwise_and(canvas, noise)

        yolo_labels = []
        num_lanes = random.randint(3, 7)
        lane_height = c_h // num_lanes
        
        for i in range(num_lanes):
            lane_y = i * lane_height + lane_height // 2
            row_x = random.randint(60, 200)
            char_size = random.randint(45, 65)
            row_y = lane_y + random.randint(-20, 20)
            
            mode = random.randint(0, 1)

            if mode == 1: # --- 分数 ---
                n_len, d_len = random.randint(1, 4), random.randint(1, 3)
                bar_w = max(n_len, d_len) * (char_size + 8) + 50
                bar_y = row_y
                bar_h = random.randint(5, 9)
                
                # barの描画とラベル
                cv2.rectangle(canvas, (row_x, bar_y), (row_x + bar_w, bar_y + bar_h), 0, -1)
                yolo_labels.append((class_to_id["bar"], (row_x, bar_y, bar_w, bar_h)))
                
                # 文字を書き込み
                for j, c in enumerate(random.choices(["var_x", "1", "2", "3", "plus"], k=n_len)):
                    paste_char(canvas, c, row_x + 20 + j*(char_size+5), bar_y - char_size - 15, char_size)
                for j, c in enumerate(random.choices(["2", "3", "5", "var_x"], k=d_len)):
                    paste_char(canvas, c, row_x + 20 + j*(char_size+5), bar_y + bar_h + 15, char_size)
                
                
                # 文字がはみ出さないよう、上下左右に大きめのマージンをとる
                m = char_size // 3
                fx, fy = row_x - m, bar_y - char_size - 25 - m
                fw, fh = bar_w + 2*m, (char_size * 2) + bar_h + 50 + 2*m
                yolo_labels.append((class_to_id["fraction"], (fx, fy, fw, fh)))
                
                # 後続の式
                eq_chars = ["equal", random.choice(["0", "1"])]
                ex_start = row_x + bar_w + 20
                for c in eq_chars:
                    paste_char(canvas, c, ex_start, bar_y - char_size//2, char_size)
                    ex_start += char_size + 15
                # equationラベルにもマージン
                yolo_labels.append((class_to_id["equation"], 
                                   (row_x + bar_w + 10, bar_y - char_size//2 - 15, 
                                    (char_size + 20) * len(eq_chars) + 10, char_size + 30)))

            else: # --- 方程式 ---
                eq = random.choices(["1","2","3","var_x"], k=1) + ["plus"] + \
                     random.choices(["1","5","var_x"], k=1) + ["equal", "0"]
                
                start_x = row_x
                for c in eq:
                    char_y = row_y + random.randint(-5, 5)
                    paste_char(canvas, c, row_x, char_y, char_size)
                    row_x += char_size + random.randint(12, 28)
                
                m_x, m_y = 20, 20
                yolo_labels.append((class_to_id["equation"], 
                                   (start_x - m_x, row_y - m_y, (row_x - start_x) + 2*m_x, char_size + 2*m_y)))

        # 画像とラベルの保存
        img_name = f"real_{idx:05d}"
        cv2.imwrite(str(output_path / "images" / f"{img_name}.jpg"), canvas)
        with open(output_path / "labels" / f"{img_name}.txt", "w") as f:
            for cls_id, (bx, by, bw, bh) in yolo_labels:
                f.write(f"{cls_id} {(bx+bw/2)/c_w:.6f} {(by+bh/2)/c_h:.6f} {bw/c_w:.6f} {bh/c_h:.6f}\n")

    print(f"\nデータ生成完了！")

if __name__ == "__main__":
    generate_mixed_data()

In [ ]:
import os
import shutil
import random

# --- 設定 ---
IMG_DIR = r"yolo_synthetic_data"
LBL_DIR = "label"
DEST_DIR = "dataset"
RATIO = 0.8  # 80%を学習用、20%を検証用

def split_dataset():
    images = [f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(images)
    
    split_idx = int(len(images) * RATIO)
    train_imgs = images[:split_idx]
    val_imgs = images[split_idx:]

    for split, img_list in [("train", train_imgs), ("val", val_imgs)]:
        img_dest = os.path.join(DEST_DIR, split, "images")
        lbl_dest = os.path.join(DEST_DIR, split, "labels")
        os.makedirs(img_dest, exist_ok=True)
        os.makedirs(lbl_dest, exist_ok=True)

        for img_name in img_list:
            lbl_name = os.path.splitext(img_name)[0] + ".txt"
            # 画像コピー
            shutil.copy(os.path.join(IMG_DIR, img_name), os.path.join(img_dest, img_name))
            # 対応するラベルをコピー
            if os.path.exists(os.path.join(LBL_DIR, lbl_name)):
                shutil.copy(os.path.join(LBL_DIR, lbl_name), os.path.join(lbl_dest, lbl_name))

    print("データ分割完了！")

split_dataset()

In [ ]:
from ultralytics import YOLO
import os
from pathlib import Path
import torch

def main():
    # --- パス設定 ---
    dataset_dir = Path(r"yolo_synthetic_data")
    yaml_path = Path(r"C:SHIKIdata.yaml")
    
    # --- 1. data.yaml の自動作成 (構造検出用 3クラス) ---
    class_names = ["equation", "fraction", "bar"] 
    
    yaml_content = f"""
path: {str(dataset_dir.absolute())}  # データセットのルート
train: images  # 学習画像フォルダ
val: images    # 検証画像

nc: {len(class_names)}
names: {class_names}
"""
    with open(yaml_path, "w", encoding="utf-8") as f:
        f.write(yaml_content.strip())
    print(f"{yaml_path} を作成しました。")

    # --- 2. モデルのロード ---
    # yolov8n.pt をベースに転移学習を行います
    model = YOLO("yolov8n.pt") 

    # --- 3. 学習の実行 ---
    print("数式の構造検出の学習を開始します...")
    
    # GPU（CUDA）が使用可能か自動判別
    device_id = 0 if torch.cuda.is_available() else 'cpu'
    print(f"使用デバイス: {device_id}")

    results = model.train(
        data=str(yaml_path),
        epochs=50,             # 3000枚あるため50エポックで十分な精度が期待できます
        imgsz=640,             # 画像サイズ。手書き構造の検出に適した標準サイズ
        device=device_id,      # CUDA環境なら0、それ以外ならcpu
        workers=0,             # Windowsのマルチプロセスエラー回避用
        batch=16,              # メモリ不足なら8に下げてください
        name="math_structure_model", # 保存名：runs/detect/math_structure_model
        exist_ok=True          # フォルダを上書き
    )
    
    print("\n学習が完了しました！")
    print(f"最高モデルのパス: runs/detect/math_structure_model/weights/best.pt")

if __name__ == "__main__":
    main()

In [ ]:
#テスト
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

# --- 設定 ---
# 学習結果の重みファイルのパス（デフォルトでは runs/detect/math_pro_model/weights/best.pt にあるはずです）
MODEL_PATH = r"shiki_structure_yolo_v8.pt" 
# テストしたい式の画像パス
TEST_IMAGE = r"" 

def run_test():
    # 1. モデルのロード
    model = YOLO(MODEL_PATH)

    # 2. 推論の実行
    # conf=0.25 は「確信度が25%以上のものだけ表示」という意味です
    results = model.predict(source=TEST_IMAGE, conf=0.25, save=True)

    # 3. 結果の表示
    # results[0].plot() で、枠と名前が書き込まれた画像が取得できます
    res_img = results[0].plot()
    res_img_rgb = cv2.cvtColor(res_img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10, 10))
    plt.imshow(res_img_rgb)
    plt.axis('off')
    plt.show()

    print(f"結果を保存しました: {results[0].save_dir}")

if __name__ == "__main__":
    run_test()